In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import *
from dataclasses import dataclass
from delta.tables import DeltaTable

In [0]:
from dataclasses import dataclass
from pyspark.sql import DataFrame
from pyspark.sql.functions import col, current_date, current_timestamp


@dataclass
class Bronze:
    header: bool
    delimiter: str
    file_path: str
    table_name: str
    file_format: str = "csv"

    def __post_init__(self) -> None:
        self.table_name_bronze = f"{self.table_name}_bronze"

    @classmethod
    def from_conf(cls, pipeline_name: str):
        conf = (
            spark.table("proj.aviation.table_config")
            .filter(col("pipeline_name") == pipeline_name)
            .first()
        )
        if not conf:
            raise ValueError(f"Pipeline '{pipeline_name}' not found in proj.aviation.table_config.")

        return cls(
            header=conf.header.lower() == "true" if isinstance(conf.header, str) else bool(conf.header),
            delimiter=conf.delimiter,
            file_path=conf.file_path,
            table_name=conf.table_name,
            file_format=getattr(conf, "file_format", "csv"),
        )

    def get_unprocessed_files(self) -> list[str]:
        raw_files = [
            f.path for f in dbutils.fs.ls(self.file_path)
            if f.name.endswith(f".{self.file_format}")
        ]

        if not spark.catalog.tableExists(self.table_name_bronze):
            return raw_files

        processed_files = {
            row._file_path
            for row in spark.read.table(self.table_name_bronze).select("_file_path").distinct().collect()
        }

        return [path for path in raw_files if path not in processed_files]

    def read(self, execution_path: str = None) -> DataFrame:
        if execution_path:
            target_paths = [execution_path]
        else:
            target_paths = self.get_unprocessed_files()

        if not target_paths:
            return None

        df = (
            spark.read.format(self.file_format)
            .option("header", self.header)
            .option("delimiter", self.delimiter)
            .load(target_paths)
        )

        return (
            df.withColumn("_load_dt", current_date())
            .withColumn("_load_dttm", current_timestamp())
            .withColumn("_file_name", col("_metadata.file_name"))
            .withColumn("_file_path", col("_metadata.file_path"))
            .withColumn("_file_size", col("_metadata.file_size"))
            .withColumn("_file_modification_time", col("_metadata.file_modification_time"))
        )

    def load_to_bronze_table(self, df: DataFrame) -> str:
        if df is None:
            return f"No new files to ingest for {self.table_name_bronze}. Process skipped."

        (
            df.write.mode("append")
            .format("delta")
            .saveAsTable(self.table_name_bronze)
        )
        return f"{self.table_name_bronze} appended successfully"

In [0]:
@dataclass
class Silver:
    table_name: str
    schema_details: dict[str, str]
    keys: list[str]
    mode: str

    def __post_init__(self) -> None:
        self.table_name_bronze = f"{self.table_name}_bronze"
        self.table_name_silver = f"{self.table_name}_silver"
        self.table_name_bad_rec = f"{self.table_name}_bad_rec"
        self.data_col = list(self.schema_details.keys())

    @classmethod
    def from_conf(
        cls, pipeline_name: str, config_table: str = "proj.aviation.table_config"
    ):
        conf = (
            spark.table(config_table)
            .filter(col("pipeline_name") == pipeline_name)
            .first()
        )
        if not conf:
            raise ValueError(
                f"Pipeline '{pipeline_name}' not found in {config_table}."
            )

        return cls(
            table_name=conf.table_name,
            schema_details=conf.schema_details,
            keys=conf.keys,
            mode=conf.mode,
        )

    def read_from_bronze(self) -> DataFrame:
        bronze_df = spark.table(self.table_name_bronze).select(*self.data_col)
        return bronze_df.withColumn(
            "_sk", sha2(concat_ws("||", *self.keys), 256)
        )

    def validate_records(self, df: DataFrame) -> DataFrame:
        reasons = []

    
        for k in self.keys:
            reasons.append(
                when(col(k).isNull(), lit(f"key_null_{k}")).otherwise(lit(None))
            )

       
        for col_name, target_type in self.schema_details.items():
            if target_type.lower() == "int":
                cast_col = col(col_name).cast("float").cast("int")
            else:
                cast_col = col(col_name).cast(target_type)

            is_corrupted = col(col_name).isNotNull() & cast_col.isNull()
            reasons.append(
                when(
                    is_corrupted, lit(f"invalid_{target_type}_{col_name}")
                ).otherwise(lit(None))
            )

        return df.withColumn("_reasons", array_compact(array(*reasons)))

    def tag_duplicates(self, df: DataFrame) -> DataFrame:
        
        partition_key = Window.partitionBy(*self.keys)
        df_with_rn = df.withColumn("_dup_cnt", count("*").over(partition_key))

        
        return df_with_rn.withColumn(
            "_reasons",
            when(
                col("_dup_cnt") > 1,
                concat(col("_reasons"), array(lit("duplicate_key"))),
            ).otherwise(col("_reasons")),
        ).drop("_dup_cnt")

    def split_good_and_bad(
        self, df: DataFrame
    ) -> tuple[DataFrame, DataFrame]:
        
        bad_df = df.filter(size(col("_reasons")) > 0)

        
        good_df = df.filter(size(col("_reasons")) == 0).drop("_reasons", "_sk")

        
        cast_exprs = []
        for c, t in self.schema_details.items():
            if t.lower() == "int":
                cast_exprs.append(col(c).cast("float").cast("int").alias(c))
            else:
                cast_exprs.append(col(c).cast(t).alias(c))

        good_casted_df = good_df.select(*cast_exprs)

     
        good_clean_df = good_casted_df.withColumn(
            "CancellationCode",
            when(col("Cancelled") == 0, lit("NOT_CANCELLED")).otherwise(
                when(
                    col("CancellationCode").isNotNull(), col("CancellationCode")
                ).otherwise(lit("UNKNOWN"))
            ),
        )

       
        delay_causes = [
            "CarrierDelay",
            "WeatherDelay",
            "NASDelay",
            "SecurityDelay",
            "LateAircraftDelay",
        ]
        is_completed_flight = (col("Cancelled") == 0) & (col("Diverted") == 0)

        for cause in delay_causes:
            if cause in self.data_col:
                good_clean_df = good_clean_df.withColumn(
                    cause,
                    when(
                        is_completed_flight,
                        when(col(cause).isNotNull(), col(cause)).otherwise(
                            lit(0.0)
                        ),
                    ).otherwise(lit(None)),
                )

        final_good_df = good_clean_df.select(
            "*",
            current_date().alias("_load_dt"),
            current_timestamp().alias("_load_dttm"),
        )

        return final_good_df, bad_df

    def load_silver(
        self, good_df: DataFrame, bad_df: DataFrame
    ) -> tuple[str, str]:
        
        (
            bad_df.write.mode(self.mode)
            .format("delta")
            .saveAsTable(self.table_name_bad_rec)
        )

       
        if not spark.catalog.tableExists(self.table_name_silver):
            
            (
                good_df.write.mode(self.mode)
                .format("delta")
                .saveAsTable(self.table_name_silver)
            )
        else:
           
            silver_delta = DeltaTable.forName(spark, self.table_name_silver)
            match_cond = " AND ".join([f"target.{k} = source.{k}" for k in self.keys])

            (
                silver_delta.alias("target")
                .merge(good_df.alias("source"), condition=match_cond)
                .whenMatchedUpdateAll()
                .whenNotMatchedInsertAll()
                .execute()
            )

        return (
            f"{self.table_name_silver} loaded (SCD 1) successfully",
            f"{self.table_name_bad_rec} appended successfully",
        )